# 11.17 - RAG Synthesis & Review

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Combine every technique from this phase into one 'support agent': ingestion, chunking, keyword + vector hybrid, reranking, grounded generation, and evaluation. This is the mini-project pattern - a production-style pipeline with citations and abstention.

## 2. Why Does This Matter?

Knowing components isn't enough; you must assemble them into one system and walk through its trace, measuring retrieval and generation independently.

## 3. Prerequisites

Units 11.1-11.16.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build one 'support agent' combining vector + keyword + rerank + grounded generation + eval
- Walk through a full pipeline trace for a sample query
- Reflect on design decisions and failure modes

## 5. Mental Model

The full RAG stack is a factory: documents enter, grounded, cited answers come out - and every stage is measured.

```text
Docs -> split -> embed_index -> query -> hybrid(ferank) -> build_context -> llm -> answer
```


## 6. Setup
Embedding + LLM helpers (both offline-safe).

In [1]:
# Deterministic embedding helper.
# Loads all-MiniLM-L6-v2 if available; otherwise falls back to a hash-based
# vector so every cell still completes offline. The fallback still gives
# "similar text -> similar vector" behaviour via character-bigram overlap,
# so the demos remain meaningful without the model download.
import hashlib, numpy as np

_DIM = 384


def _hash_embed(texts):
    vecs = np.zeros((len(texts), _DIM))
    for i, t in enumerate(texts):
        bigrams = [t[j:j+2].lower() for j in range(len(t)-1)]
        for bg in bigrams:
            h = int(hashlib.md5(bg.encode()).hexdigest(), 16) % _DIM
            vecs[i, h] += 1.0
        norm = np.linalg.norm(vecs[i]) or 1.0
        vecs[i] = vecs[i] / norm
    return vecs


_model = None
_model_name = "all-MiniLM-L6-v2"


def get_embedder(force_fallback=False):
    """Return a function texts -> np.ndarray (N, dim)."""
    global _model
    if force_fallback:
        return _hash_embed
    if _model is None:
        try:
            from sentence_transformers import SentenceTransformer
            _model = SentenceTransformer(_model_name)
        except Exception as e:
            print("MiniLM unavailable, using hash fallback:", type(e).__name__)
            _model = None
    if _model is None:
        return _hash_embed
    return lambda texts: np.asarray(_model.encode(list(texts), convert_to_numpy=True))


def embed(texts, force_fallback=False):
    fn = get_embedder(force_fallback=force_fallback)
    return np.asarray(fn(texts), dtype=np.float32)


print("embedding dim:", _DIM)
print("backend:", _model_name if get_embedder() != _hash_embed else "hash-fallback")


embedding dim: 384


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6283.46it/s]

backend: all-MiniLM-L6-v2


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
import operator

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. Corpus + Index
A support FAQ across a few topics. We build the Chroma vector index and keep the raw text for keyword/BM25 fusion.

In [3]:
import chromadb

docs = [
    "Returns are accepted within 30 days of purchase in original packaging.",
    "Return shipping is paid by the customer unless the item is defective.",
    "Standard shipping takes 5-7 business days; express 2-3 days.",
    "Free standard shipping applies to US orders over fifty dollars.",
    "Refunds post to the original payment method within 5-7 business days.",
    "The one-year warranty covers manufacturing defects but not accidental damage.",
]
sources = ["policy", "policy", "shipping", "shipping", "refund", "warranty"]
client = chromadb.Client()
col = client.create_collection("support_agent", embedding_function=None,
                               metadata={"hnsw:space": "cosine"})
emb = embed(docs)
col.add(documents=docs, embeddings=emb.tolist(), ids=[f"c{i}" for i in range(len(docs))],
        metadatas=[{"source": s} for s in sources])
print("indexed", col.count(), "chunks")


indexed 6 chunks


## 8. Components: BM25 + Vector + Rerank
Combine a keyword score with vector rank via RRF, then rerank the shortlist with a lexical+cosine blend to choose the best evidence.

In [4]:
import numpy as np

STOP = {"the", "a", "an", "to", "of", "in", "within", "and", "is", "for"}


def tok(s):
    return [w for w in s.lower().replace(".", "").replace(",", "").split() if w not in STOP]


def bm25(query):
    qt = tok(query)
    n = len(docs)
    avg = np.mean([len(tok(d)) for d in docs])
    scores = np.zeros(n)
    for d_i, doc in enumerate(docs):
        dt = tok(doc)
        for t in qt:
            tf_t = dt.count(t)
            if tf_t == 0:
                continue
            df = sum(1 for dd in docs if t in tok(dd))
            idf = np.log(1 + (n - df + 0.5) / (df + 0.5))
            denom = tf_t + 1.5 * (1 - 0.75 + 0.75 * len(dt) / avg)
            scores[d_i] += idf * tf_t * 2.5 / denom
    return scores


def ranked_candidates(query, take=10):
    bm = bm25(query)
    vec = col.query(query_embeddings=embed([query]).tolist(),
                    n_results=len(docs))["ids"][0]
    bm_order = list(np.argsort(-bm))
    vec_order = [int(i[1:]) for i in vec]
    fused = {}
    for rank, idx in enumerate(bm_order):
        fused[idx] = fused.get(idx, 0) + 1.0 / (60 + rank + 1)
    for rank, idx in enumerate(vec_order):
        fused[idx] = fused.get(idx, 0) + 1.0 / (60 + rank + 1)
    ordered = [i for i, _ in sorted(fused.items(), key=lambda x: -x[1])]
    return ordered[:take]


def choose_best(query, cands):
    def score(i):
        overlap = len(set(tok(query)) & set(tok(docs[i]))) / max(1, len(set(tok(query))))
        cos = float(np.dot(embed([query])[0], embed([docs[i]])[0]))
        return overlap + cos
    return max(cands, key=score)


## 9. Grounded Generation (support agent answer)
Turn the chosen evidence into a grounded, cited answer using llm(). The prompt forces cite-only-from-context and abstention.

In [5]:
def support_answer(query, k=4):
    cands = ranked_candidates(query, take=k)
    best = choose_best(query, cands)
    ctx = f"[1] ({sources[best]})\n{docs[best]}"
    prompt = (
        "Answer ONLY from the context and cite [1]. If it doesn't answer, say you don't know.\n"
        f"Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer:"
    )
    return best, sources[best], docs[best], llm(prompt)


## 10. Full Pipeline Trace
Walk a query end-to-end and print every stage: candidates, chosen evidence, source, and the grounded answer.

In [6]:
query = "How fast will my refund arrive and who pays return shipping?"

bm = bm25(query)
print("BM25 scores     :", [round(float(x), 3) for x in bm])
vec_order = [int(i[1:]) for i in col.query(query_embeddings=embed([query]).tolist(), n_results=len(docs))["ids"][0]]
print("vector ranks    :", vec_order)
cands = ranked_candidates(query, take=4)
print("fused candidates:", cands)
best, src, chunk, ans = support_answer(query)
print("chosen evidence :", best, f"({src})", "->", chunk)
print("ANSWER          :", ans)


BM25 scores     : [0.0, 1.582, 0.0, 0.0, 0.0, 0.0]
vector ranks    : [4, 1, 0, 2, 3, 5]


fused candidates: [np.int64(1), np.int64(0), np.int64(4), np.int64(2)]


chosen evidence : 1 (policy) -> Return shipping is paid by the customer unless the item is defective.
ANSWER          : Return shipping is paid by the customer unless the item is defective [1]. I don't know how fast the refund will arrive.


## 11. Simple Evaluation on the Support Agent
Retrieve for gold queries and measure whether the intended doc makes the top-k. This is the scalable evaluation harness you'd grow in production.

In [7]:
gold = {
    "how do i return a damaged product": 0,
    "how fast is standard shipping": 2,
    "is accidental damage covered by warranty": 5,
    "when will i see my refund": 4,
}
correct = 0
for q, gold_idx in gold.items():
    cands = ranked_candidates(q, take=4)
    hit = gold_idx in cands
    correct += int(hit)
    print(f"{q[:40]:40s} recall: {hit} (cands {cands})")
print(f"\ntop-k recall: {correct}/{len(gold)} = {correct/len(gold):.2f}")


how do i return a damaged product        recall: True (cands [np.int64(1), np.int64(0), np.int64(2), np.int64(5)])


how fast is standard shipping            recall: True (cands [np.int64(3), np.int64(2), np.int64(1), np.int64(0)])


is accidental damage covered by warranty recall: True (cands [np.int64(5), np.int64(1), np.int64(0), np.int64(2)])


when will i see my refund                recall: True (cands [np.int64(0), np.int64(1), np.int64(4), np.int64(2)])

top-k recall: 4/4 = 1.00


## 12. Reflection Questions
Now that the stack is assembled, reason about the system:

1. If the answer is right but sources are wrong - which stage failed?
2. If the right source exists but isn't retrieved (low recall) - what levers do you pull first?
3. If the answer invents facts - is that a retrieval or generation bug?
4. When would you add agentic routing (11.16) on top of this fixed pipeline?
5. How would you extend to thousands of documents (persistence, indexing, latency)?

In [8]:
# Quick manual check you can re-run after tuning any component.
q = "does the warranty cover a broken screen from a drop?"
cands = ranked_candidates(q, take=4)
print("candidates for warranty q:", cands, "->", [docs[i][:40] for i in cands])


candidates for warranty q: [np.int64(5), np.int64(0), np.int64(1), np.int64(2)] -> ['The one-year warranty covers manufacturi', 'Returns are accepted within 30 days of p', 'Return shipping is paid by the customer ', 'Standard shipping takes 5-7 business day']



## Common Mistakes

- Retrieval issues and generation issues treated as one problem.
- Not measuring retrieval separately from generation.
- Changing everything at once (can't tell what fixed it).
- Forgetting abstractions for unanswerable / out-of-scope queries.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Right answer, wrong sources | Context/re-rank issue | Inspect candidates, fix fusion |
| Right source exists but not retrieved | Low recall | Increase k, improve embedding/re-rank |
| Answer invents facts | Weak grounding | Require citations + abstention |
| Slow pipeline | Too many candidate scans | Reduce candidate set, batch embeds |

## Best Practices

- Establish a naive baseline and improve one layer at a time.
- Log candidates AND the final prompt for every answer.
- Always instruct cite-only-from-context and abstention.
- Keep an eval set from day one; track metrics over time.
- Balance latency/cost against quality deliberately.

## Hands-On Practice

1. **Basic:** Re-run the support agent on new queries and read the trace.
2. **Guided:** Add metadata filtering by source to the agent.
3. **Independent:** Grow the eval set to 10+ queries and re-measure recall.
4. **Realistic:** Replace the mock llm() with a real grounded call when a key is set.
5. **Challenge:** Add agentic routing (11.16) so the agent decides when to retrieve.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
